In [ ]:
from pyspark.sql import SparkSession

spark=SparkSession.builder \
.appName("Milestone Assessment 2") \
.getOrCreate()

In [ ]:
%%writefile stores.csv
store_id,store_name,city,state,store_type,manager_name
S101,Metro Mart Hyderabad,Hyderabad,Telangana,Supermarket,Rahul Sharma
S102,Metro Mart Bangalore,Bangalore,Karnataka,Supermarket,Priya Reddy
S103,Metro Mart Mumbai,Mumbai,Maharashtra,Hypermarket,Amit Kumar
S104,Metro Mart Chennai,Chennai,Tamil Nadu,Supermarket,Sneha Patel
S105,Metro Mart Delhi,Delhi,Delhi,Hypermarket,Farhan Ali
S106,Metro Mart Pune,Pune,Maharashtra,Mini Store,Neha Singh
S107,Metro Mart Kochi,Kochi,Kerala,Mini Store,Arjun Verma
S108,Metro Mart Jaipur,Jaipur,Rajasthan,Supermarket,Meera Nair

In [ ]:
%%writefile products.csv
product_id,product_name,category,brand,supplier_id,unit_price
P101,Laptop,Electronics,Lenovo,S201,65000
P102,Mobile,Electronics,Samsung,S202,25000
P103,Television,Electronics,LG,S203,45000
P104,Office Chair,Furniture,Featherlite,S204,7000
P105,Study Table,Furniture,Urban Ladder,S204,12000
P106,Shoes,Fashion,Nike,S205,4500
P107,Watch,Fashion,Fastrack,S206,8000
P108,Backpack,Fashion,Wildcraft,S206,2500
P109,Refrigerator,Electronics,Whirlpool,S203,38000
P110,Sofa,Furniture,Godrej,S204,32000
P111,Headphones,Electronics,Sony,S999,3000
P112,T-Shirt,Fashion,Puma,,1500

In [ ]:
%%writefile inventory.csv
inventory_id,store_id,product_id,stock_quantity,reorder_level,last_update
I1001,S101,P101,10,5,2026-01-10
I1002,S101,P102,25,10,2026-01-10
I1003,S101,P104,3,5,2026-01-11
I1004,S102,P101,8,5,2026-01-12
I1005,S102,P103,5,4,2026-01-12
I1006,S103,P105,2,5,2026-01-13
I1007,S103,P106,30,10,2026-01-14
I1008,S104,P107,4,5,2026-01-15
I1009,S105,P108,50,20,2026-01-15
I1010,S106,P109,,6,2026-01-16
I1011,S107,P110,1,3,2026-01-17
I1012,S108,P120,12,5,2026-01-18

In [ ]:
%%writefile sales.csv
sale_id,store_id,product_id,sale_date,quantity_sold,sale_amount,payment_m
SA1001,S101,P101,2026-01-10,1,65000,UPI
SA1002,S101,P102,2026-01-10,2,50000,Card
SA1003,S102,P101,2026-01-11,1,65000,UPI
SA1004,S103,P106,2026-01-12,4,18000,Cash
SA1005,S104,P107,2026-01-12,1,8000,Card
SA1006,S105,P108,2026-01-13,5,12500,UPI
SA1007,S106,P109,2026-01-14,1,38000,Card
SA1008,S107,P110,2026-01-15,1,32000,UPI
SA1009,S108,P120,2026-01-15,2,10000,Cash
SA1010,S101,P104,2026-01-16,2,14000,
SA1011,S102,P103,2026-01-17,1,,UPI
SA1012,S103,P105,2026-01-18,1,12000,Card
SA1013,S104,P107,2026-02-01,2,16000,UPI
SA1014,S105,P108,2026-02-02,3,7500,Cash
SA1015,S101,P102,2026-02-03,1,25000,Card

In [ ]:
%%writefile suppliers.json
[
{
"supplier_id": "S201",
"supplier_name": "TechSource India",
"city": "Hyderabad",
"rating": 4.5,
"contact": {
"phone": "9876500011",
"email": "techsource@mail.com"
}
},
{
"supplier_id": "S202",
"supplier_name": "MobileWorld Distributors",
"city": "Bangalore",
"rating": 4.2,
"contact": {
"phone": null,
"email": "mobileworld@mail.com"
}
},
{
"supplier_id": "S203",
"supplier_name": "HomeTech Supply",
"city": "Mumbai",
"rating": 4.4,
"contact": {
"phone": "9876500013",
"email": null
}
},
{
"supplier_id": "S204",
"supplier_name": "Urban Furniture Co",
"city": "Delhi",
"rating": 4.0,
"contact": {
"phone": "9876500014",
"email": "urban@mail.com"
}
},
{
"supplier_id": "S205",
"supplier_name": "Fashion Direct",
"city": "Pune",
"rating": 3.8,
"contact": {
"phone": null,
"email": null
}
}
]

In [ ]:
#1
stores_df = spark.read.option("header", True).option("inferSchema", True).csv("stores.csv")
stores_df.show()

In [ ]:
#2
products_df = spark.read.option("header", True).option("inferSchema", True).csv("products.csv")
products_df.show()

In [ ]:
#3
inventory_df = spark.read.option("header", True).option("inferSchema", True).csv("inventory.csv")
inventory_df.show()

In [ ]:
#4
sales_df = spark.read.option("header", True).option("inferSchema", True).csv("sales.csv")
sales_df = sales_df.withColumnRenamed("payment_m", "payment_mode")
sales_df.show()

In [ ]:
#5
suppliers_df = spark.read.option("multiline", True).json("suppliers.json")
suppliers_df.show(truncate=False)

In [ ]:
#6
stores_df.printSchema()
products_df.printSchema()
inventory_df.printSchema()
sales_df.printSchema()
suppliers_df.printSchema()

In [ ]:
#7
print(stores_df.count())
print(products_df.count())
print(inventory_df.count())
print(sales_df.count())
print(suppliers_df.count())

In [ ]:
#8
stores_df.write.mode("overwrite").parquet("bronze/stores")
products_df.write.mode("overwrite").parquet("bronze/products")
inventory_df.write.mode("overwrite").parquet("bronze/inventory")
sales_df.write.mode("overwrite").parquet("bronze/sales")
suppliers_df.write.mode("overwrite").parquet("bronze/suppliers")

In [ ]:
#9
products_df.filter(
    products_df.supplier_id.isNull() |
    (products_df.supplier_id == "")
).show()

In [ ]:
#10
inventory_df.filter(
    inventory_df.stock_quantity.isNull()
).show()

In [ ]:
#11
sales_df.filter(
    sales_df.sale_amount.isNull()
).show()

In [ ]:
#12
sales_df.filter(
    sales_df.payment_mode.isNull() |
    (sales_df.payment_mode == "")
).show()

In [ ]:
#13
inventory_df = inventory_df.fillna(
    {"stock_quantity": 0}
)

In [ ]:
#14
sales_df = sales_df.fillna(
    {"sale_amount": 0}
)

In [ ]:
#15
sales_df = sales_df.fillna(
    {"payment_mode": "Not Provided"}
)

In [ ]:
#16
products_df = products_df.fillna(
    {"supplier_id": "UNKNOWN"}
)

In [ ]:
#17
from pyspark.sql.functions import when, col

products_df = products_df.withColumn(
    "data_quality_status",
    when(
        (col("supplier_id").isNull()) | (col("supplier_id") == ""),
        "Invalid"
    ).otherwise("Valid")
)

inventory_df = inventory_df.withColumn(
    "data_quality_status",
    when(
        col("stock_quantity").isNull(),
        "Invalid"
    ).otherwise("Valid")
)

sales_df = sales_df.withColumn(
    "data_quality_status",
    when(
        col("sale_amount").isNull(),
        "Invalid"
    ).otherwise("Valid")
)

In [ ]:
#18
products_df.write.mode("overwrite").parquet("silver/products")

inventory_df.write.mode("overwrite").parquet("silver/inventory")

sales_df.write.mode("overwrite").parquet("silver/sales")

In [ ]:
#19
from pyspark.sql.functions import col

suppliers_flat_df = suppliers_df.select(
    "supplier_id",
    "supplier_name",
    "city",
    "rating",
    col("contact.phone").alias("phone"),
    col("contact.email").alias("email")
)

suppliers_flat_df.show()

In [ ]:
#20
suppliers_flat_df.select(
    "supplier_id",
    "phone"
).show()

In [ ]:
#21
suppliers_flat_df.select(
    "supplier_id",
    "email"
).show()

In [ ]:
#22
suppliers_flat_df = suppliers_flat_df.fillna(
    {"phone": "Not Provided"}
)

In [ ]:
#23
suppliers_flat_df = suppliers_flat_df.fillna(
    {"email": "Not Provided"}
)

In [ ]:
#24
suppliers_flat_df.write.mode("overwrite").parquet(
    "silver/suppliers_flat"
)

In [ ]:
#25
product_supplier_df = products_df.join(
    suppliers_flat_df,
    "supplier_id",
    "left"
)

product_supplier_df.show()

In [ ]:
#26
inventory_product_df = inventory_df.join(
    products_df,
    "product_id",
    "left"
)

inventory_product_df.show()

In [ ]:
#27
sales_store_df = sales_df.join(
    stores_df,
    "store_id",
    "left"
)

sales_store_df.show()

In [ ]:
#28
sales_product_df = sales_df.join(
    products_df,
    "product_id",
    "left"
)

sales_product_df.show()

In [ ]:
#29
retail_sales_df = sales_df.join(
    stores_df,
    "store_id",
    "left"
).join(
    products_df,
    "product_id",
    "left"
)

retail_sales_df.show()

In [ ]:
#30
products_df.join(
    suppliers_flat_df,
    "supplier_id",
    "left_anti"
).show()

In [ ]:
#31
inventory_df.join(
    products_df,
    "product_id",
    "left_anti"
).show()

In [ ]:
#32
sales_df.join(
    products_df,
    "product_id",
    "left_anti"
).show()

In [ ]:
#33
sales_df.join(
    stores_df,
    "store_id",
    "left_anti"
).show()

In [ ]:
#34
from pyspark.sql.functions import when

inventory_df = inventory_df.withColumn(
    "stock_status",
    when(
        col("stock_quantity") <= col("reorder_level"),
        "Reorder Required"
    ).otherwise("Sufficient Stock")
)

In [ ]:
#35
products_df = products_df.withColumn(
    "price_category",
    when(col("unit_price") >= 50000, "Premium")
    .when(col("unit_price") >= 10000, "Standard")
    .otherwise("Budget")
)

In [ ]:
#36
sales_df = sales_df.withColumn(
    "revenue_category",
    when(col("sale_amount") >= 50000, "High Revenue")
    .when(col("sale_amount") >= 15000, "Medium Revenue")
    .otherwise("Low Revenue")
)

In [ ]:
#37
from pyspark.sql.functions import month

sales_df = sales_df.withColumn(
    "month",
    month("sale_date")
)

In [ ]:
#38
from pyspark.sql.functions import year

sales_df = sales_df.withColumn(
    "year",
    year("sale_date")
)

In [ ]:
#39
inventory_value_df = inventory_df.join(
    products_df,
    "product_id"
).withColumn(
    "inventory_value",
    col("stock_quantity") * col("unit_price")
)

inventory_value_df.show()

In [ ]:
#40
suppliers_flat_df = suppliers_flat_df.withColumn(
    "supplier_quality",
    when(col("rating") >= 4.5, "Excellent")
    .when(col("rating") >= 4.0, "Good")
    .otherwise("Average")
)

In [ ]:
#41
stores_df.groupBy(
    "state"
).count().show()

In [ ]:
#42
products_df.groupBy(
    "category"
).count().show()

In [ ]:
#43
products_df.groupBy(
    "brand"
).count().show()

In [ ]:
#44
inventory_value_df.groupBy(
    "store_id"
).sum("inventory_value").show()

In [ ]:
#45
inventory_value_df.groupBy(
    "category"
).sum("inventory_value").show()

In [ ]:
#46
inventory_df.filter(
    col("stock_quantity") <= col("reorder_level")
).count()

In [ ]:
#47
sales_df.agg(
    {"sale_amount": "sum"}
).show()

In [ ]:
#48
sales_df.groupBy(
    "store_id"
).sum("sale_amount").show()

In [ ]:
#49
sales_df.join(
    stores_df,
    "store_id"
).groupBy(
    "city"
).sum("sale_amount").show()

In [ ]:
#50
sales_df.join(
    products_df,
    "product_id"
).groupBy(
    "category"
).sum("sale_amount").show()

In [ ]:
#51
sales_df.join(products_df, "product_id") \
    .groupBy("product_id", "product_name") \
    .sum("sale_amount") \
    .show()

In [ ]:
#52
sales_df.groupBy("payment_mode") \
    .sum("sale_amount") \
    .show()

In [ ]:
#53
sales_df.join(products_df, "product_id") \
    .groupBy("product_id", "product_name") \
    .sum("sale_amount") \
    .orderBy("sum(sale_amount)", ascending=False) \
    .show(1)

In [ ]:
#54
sales_df.join(stores_df, "store_id") \
    .groupBy("store_id", "store_name") \
    .sum("sale_amount") \
    .orderBy("sum(sale_amount)", ascending=False) \
    .show(1)

In [ ]:
#55
sales_df.join(products_df, "product_id") \
    .groupBy("category") \
    .sum("sale_amount") \
    .orderBy("sum(sale_amount)", ascending=False) \
    .show(1)

In [ ]:
#56
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, sum

product_revenue_df = sales_df.join(products_df, "product_id") \
    .groupBy("product_id", "product_name") \
    .agg(sum("sale_amount").alias("total_revenue"))

window_spec = Window.orderBy(col("total_revenue").desc())

product_revenue_df.withColumn("rank", rank().over(window_spec)).show()

In [ ]:
#57
store_revenue_df = sales_df.join(stores_df, "store_id") \
    .groupBy("store_id", "store_name") \
    .agg(sum("sale_amount").alias("total_revenue"))

window_spec = Window.orderBy(col("total_revenue").desc())

store_revenue_df.withColumn("rank", rank().over(window_spec)).show()

In [ ]:
#58
category_product_revenue_df = sales_df.join(products_df, "product_id") \
    .groupBy("category", "product_id", "product_name") \
    .agg(sum("sale_amount").alias("total_revenue"))

window_spec = Window.partitionBy("category").orderBy(col("total_revenue").desc())

category_product_revenue_df.withColumn("rank", rank().over(window_spec)).show()

In [ ]:
#59
category_product_revenue_df.withColumn("rank", rank().over(window_spec)) \
    .filter(col("rank") == 1) \
    .show()

In [ ]:
#60
category_product_revenue_df.withColumn("rank", rank().over(window_spec)) \
    .filter(col("rank") <= 3) \
    .show()

In [ ]:
#61
state_store_revenue_df = sales_df.join(stores_df, "store_id") \
    .groupBy("state", "store_id", "store_name") \
    .agg(sum("sale_amount").alias("total_revenue"))

window_spec = Window.partitionBy("state").orderBy(col("total_revenue").desc())

state_store_revenue_df.withColumn("rank", rank().over(window_spec)) \
    .filter(col("rank") == 1) \
    .show()

In [ ]:
#62
from pyspark.sql.functions import sum

daily_revenue_df = sales_df.groupBy("sale_date") \
    .agg(sum("sale_amount").alias("daily_revenue"))

window_spec = Window.orderBy("sale_date")

daily_revenue_df.withColumn(
    "running_total",
    sum("daily_revenue").over(window_spec)
).show()

In [ ]:
#63
from pyspark.sql.functions import lag

window_spec = Window.orderBy("sale_date")

sales_df.withColumn(
    "previous_sale_amount",
    lag("sale_amount").over(window_spec)
).show()

In [ ]:
#64
from pyspark.sql.functions import lead

window_spec = Window.orderBy("sale_date")

sales_df.withColumn(
    "next_sale_amount",
    lead("sale_amount").over(window_spec)
).show()

In [ ]:
#65
window_spec = Window.partitionBy("product_id").orderBy("sale_date")

sales_df.withColumn(
    "previous_sale_amount",
    lag("sale_amount").over(window_spec)
).filter(
    col("sale_amount") > col("previous_sale_amount")
).show()

In [ ]:
#66
stores_df.createOrReplaceTempView("stores")
products_df.createOrReplaceTempView("products")
inventory_df.createOrReplaceTempView("inventory")
sales_df.createOrReplaceTempView("sales")
suppliers_flat_df.createOrReplaceTempView("suppliers")

In [ ]:
#67
spark.sql("SELECT * FROM sales").show()

In [ ]:
#68
spark.sql("""
SELECT category, COUNT(*) AS total_products
FROM products
GROUP BY category
""").show()

In [ ]:
#69
spark.sql("""
SELECT s.store_id, st.store_name, SUM(s.sale_amount) AS total_revenue
FROM sales s
JOIN stores st ON s.store_id = st.store_id
GROUP BY s.store_id, st.store_name
""").show()

In [ ]:
#70
spark.sql("""
SELECT st.city, SUM(s.sale_amount) AS total_revenue
FROM sales s
JOIN stores st ON s.store_id = st.store_id
GROUP BY st.city
""").show()

In [ ]:
#71
spark.sql("""
SELECT *
FROM inventory
WHERE stock_quantity <= reorder_level
""").show()

In [ ]:
#72
spark.sql("""
SELECT s.*
FROM sales s
LEFT JOIN products p ON s.product_id = p.product_id
WHERE p.product_id IS NULL
""").show()

In [ ]:
#73
spark.sql("""
SELECT p.*
FROM products p
LEFT JOIN suppliers sp ON p.supplier_id = sp.supplier_id
WHERE sp.supplier_id IS NULL
""").show()

In [ ]:
#74
spark.sql("""
SELECT p.product_id, p.product_name, SUM(s.sale_amount) AS total_revenue
FROM sales s
JOIN products p ON s.product_id = p.product_id
GROUP BY p.product_id, p.product_name
ORDER BY total_revenue DESC
LIMIT 5
""").show()

In [ ]:
#75
spark.sql("""
SELECT payment_mode, SUM(sale_amount) AS total_revenue
FROM sales
GROUP BY payment_mode
""").show()

In [ ]:
#76
retail_sales_df = sales_df.alias("s") \
    .join(stores_df.alias("st"), "store_id") \
    .join(products_df.alias("p"), "product_id") \
    .select(
        "sale_id",
        "store_id",
        "store_name",
        "city",
        "state",
        "product_id",
        "product_name",
        "category",
        "brand",
        "sale_date",
        "quantity_sold",
        "sale_amount",
        "payment_mode"
    )

retail_sales_df.write.mode("overwrite").parquet("gold/retail_sales")

In [ ]:
#77
sales_df.write.mode("overwrite").partitionBy("year", "month").parquet("gold/sales_partitioned")

In [ ]:
#78
march_sales_data = [
    ("SA1016", "S101", "P101", "2026-03-01", 1, 65000, "UPI"),
    ("SA1017", "S102", "P103", "2026-03-02", 2, 90000, "Card"),
    ("SA1018", "S105", "P108", "2026-03-03", 4, 10000, "Cash")
]

march_sales_df = spark.createDataFrame(
    march_sales_data,
    ["sale_id", "store_id", "product_id", "sale_date", "quantity_sold", "sale_amount", "payment_mode"]
)

march_sales_df.write.mode("overwrite").csv("incremental/march_sales", header=True)

In [ ]:
#79
incremental_sales_df = spark.read.csv("incremental/march_sales", header=True, inferSchema=True)
incremental_sales_df.show()

In [ ]:
#80
incremental_sales_df.write.mode("append").parquet("silver/sales")

In [ ]:
#81
updated_sales_df = spark.read.parquet("silver/sales")

updated_product_revenue_df = updated_sales_df.join(products_df, "product_id") \
    .groupBy("product_id", "product_name") \
    .sum("sale_amount")

updated_product_revenue_df.show()

In [ ]:
#82
updated_store_revenue_df = updated_sales_df.join(stores_df, "store_id") \
    .groupBy("store_id", "store_name") \
    .sum("sale_amount")

updated_store_revenue_df.show()

In [ ]:
#83
from pyspark.sql.functions import month, year

updated_sales_df = updated_sales_df.withColumn("month", month("sale_date")) \
    .withColumn("year", year("sale_date"))

updated_sales_df.write.mode("overwrite").partitionBy("year", "month").parquet("gold/final_sales")

In [ ]:
#84
before_count = sales_df.count()
after_count = updated_sales_df.count()

print("Before Count:", before_count)
print("After Count:", after_count)

In [ ]:
#85
from pyspark.sql.functions import count, sum
store_performance_report = updated_sales_df.join(stores_df, "store_id") \
    .groupBy("store_id", "store_name", "city", "state") \
    .agg(
        count("*").alias("total_sales"),
        sum("sale_amount").alias("total_revenue")
    )

store_performance_report.show()

In [ ]:
from pyspark.sql.functions import count, countDistinct, sum

In [ ]:
#86
product_performance_report = updated_sales_df.join(products_df, "product_id") \
    .groupBy("product_id", "product_name", "category", "brand") \
    .agg(
        sum("quantity_sold").alias("total_quantity_sold"),
        sum("sale_amount").alias("total_revenue")
    )

product_performance_report.show()

In [ ]:
#87
inventory_reorder_report = inventory_df.join(products_df, "product_id") \
    .select(
        "store_id",
        "product_id",
        "product_name",
        "stock_quantity",
        "reorder_level",
        "stock_status"
    )

inventory_reorder_report.show()

In [ ]:
#88
supplier_quality_report = suppliers_flat_df.select(
    "supplier_id",
    "supplier_name",
    "city",
    "rating",
    "supplier_quality",
    "phone",
    "email"
)

supplier_quality_report.show()

In [ ]:
#89
category_revenue_report = updated_sales_df.join(products_df, "product_id") \
    .groupBy("category") \
    .agg(
        countDistinct("product_id").alias("total_products"),
        sum("quantity_sold").alias("total_quantity_sold"),
        sum("sale_amount").alias("total_revenue")
    )

category_revenue_report.show()

In [ ]:
#90
payment_mode_report = updated_sales_df.groupBy("payment_mode") \
    .agg(
        count("*").alias("total_transactions"),
        sum("sale_amount").alias("total_revenue")
    )

payment_mode_report.show()